# groupby

> GroupBy operations for MXFrame using MAX Engine.
> Supports both CPU and GPU execution with the same code.

In [ ]:
#| default_exp groupby

In [ ]:
%pip install -e .. -q

In [ ]:
#| export
import pyarrow as pa
import pyarrow.compute as pc
import numpy as np
from typing import Union, List, Dict, Callable, Any, Tuple, Optional
from dataclasses import dataclass

from mxframe.core import MXFrame
from mxframe.ops import MXColumn, col
from mxframe.max_ops import MAXSession, DeviceType, max_sum, max_masked_sum

## Aggregation Expressions

Lazy expressions that describe aggregations to be computed per group.

In [ ]:
#| export
@dataclass
class AggExpr:
    """Lazy aggregation expression.
    
    Describes what column and operation to apply, evaluated during groupby.agg().
    """
    column: str
    op: str  # 'sum', 'mean', 'count', 'min', 'max'
    
    def __repr__(self) -> str:
        return f"{self.op}('{self.column}')"


def agg_sum(column: str) -> AggExpr:
    """Create a sum aggregation expression."""
    return AggExpr(column=column, op='sum')


def agg_mean(column: str) -> AggExpr:
    """Create a mean aggregation expression."""
    return AggExpr(column=column, op='mean')


def agg_count(column: str) -> AggExpr:
    """Create a count aggregation expression."""
    return AggExpr(column=column, op='count')


def agg_min(column: str) -> AggExpr:
    """Create a min aggregation expression."""
    return AggExpr(column=column, op='min')


def agg_max(column: str) -> AggExpr:
    """Create a max aggregation expression."""
    return AggExpr(column=column, op='max')

In [ ]:
# Test aggregation expressions
print(agg_sum('qty'))
print(agg_mean('price'))

## MXGroupBy Class

Returned by `MXFrame.group_by()`, holds the grouping keys and provides `.agg()` method.

In [ ]:
#| export
class MXGroupBy:
    """GroupBy object for MXFrame.
    
    Created by MXFrame.group_by(), provides .agg() to compute aggregations.
    Uses MAX Engine for compute on CPU or GPU.
    
    Example:
        >>> grouped = frame.group_by('flag', 'status')
        >>> result = grouped.agg(
        ...     sum_qty=agg_sum('qty'),
        ...     avg_price=agg_mean('price')
        ... )
    """
    
    def __init__(self, frame: MXFrame, keys: Tuple[str, ...], device: DeviceType = "auto"):
        self._frame = frame
        self._keys = keys
        self._device = device
        
        # Compute unique groups and group indices
        self._groups, self._group_ids, self._n_groups = self._compute_groups()
    
    def _compute_groups(self) -> Tuple[Dict[Tuple, int], np.ndarray, int]:
        """Compute unique group keys and per-row group IDs.
        
        Returns:
            groups: Dict mapping group tuple -> group_id
            group_ids: Array of group_id for each row
            n_groups: Number of unique groups
        """
        # Get key columns as numpy arrays
        key_arrays = [self._frame.column_to_numpy(k) for k in self._keys]
        
        # Find unique combinations
        n_rows = self._frame.num_rows
        groups: Dict[Tuple, int] = {}
        group_ids = np.zeros(n_rows, dtype=np.int32)
        
        for i in range(n_rows):
            key = tuple(arr[i] for arr in key_arrays)
            if key not in groups:
                groups[key] = len(groups)
            group_ids[i] = groups[key]
        
        return groups, group_ids, len(groups)
    
    @property
    def n_groups(self) -> int:
        """Number of unique groups."""
        return self._n_groups
    
    @property
    def keys(self) -> Tuple[str, ...]:
        """Grouping key column names."""
        return self._keys
    
    def agg(self, device: Optional[DeviceType] = None, **kwargs: AggExpr) -> MXFrame:
        """Compute aggregations per group using MAX Engine.
        
        Args:
            device: Override device selection ('cpu', 'gpu', 'auto')
            **kwargs: output_name=AggExpr pairs
            
        Returns:
            MXFrame with one row per group
        """
        device = device or self._device
        
        # Build result columns
        result_data = {}
        
        # Add group key columns
        group_keys_list = list(self._groups.keys())
        for i, key_name in enumerate(self._keys):
            result_data[key_name] = [group_keys_list[g][i] for g in range(self._n_groups)]
        
        # Compute each aggregation
        for output_name, expr in kwargs.items():
            values = self._frame.column_to_numpy(expr.column).astype(np.float32)
            result_data[output_name] = self._compute_agg(expr.op, values, device)
        
        return MXFrame(result_data)
    
    def _compute_agg(self, op: str, values: np.ndarray, device: DeviceType) -> List[float]:
        """Compute aggregation per group using MAX Engine.
        
        Uses mask-per-group approach: for each group, create a boolean mask
        and call max_masked_sum.
        """
        results = []
        
        for g in range(self._n_groups):
            mask = (self._group_ids == g).astype(np.float32)
            
            if op == 'sum':
                result = max_masked_sum(mask, values, device=device)
            elif op == 'count':
                # Count is just sum of mask
                result = max_sum(mask, device=device)
            elif op == 'mean':
                total = max_masked_sum(mask, values, device=device)
                count = max_sum(mask, device=device)
                result = total / count if count > 0 else 0.0
            elif op == 'min':
                # Fallback to PyArrow for min/max (TODO: add to MAX)
                masked_vals = values[self._group_ids == g]
                result = float(np.min(masked_vals)) if len(masked_vals) > 0 else float('nan')
            elif op == 'max':
                masked_vals = values[self._group_ids == g]
                result = float(np.max(masked_vals)) if len(masked_vals) > 0 else float('nan')
            else:
                raise ValueError(f"Unknown aggregation: {op}")
            
            results.append(result)
        
        return results
    
    def __repr__(self) -> str:
        return f"MXGroupBy(keys={self._keys}, n_groups={self._n_groups})"

In [ ]:
# Test MXGroupBy
frame = MXFrame({
    'flag': ['A', 'A', 'B', 'B', 'A'],
    'status': ['X', 'X', 'X', 'Y', 'Y'],
    'qty': [10, 20, 30, 40, 50],
    'price': [1.0, 2.0, 3.0, 4.0, 5.0],
})

grouped = MXGroupBy(frame, ('flag', 'status'))
print(f"Groups: {grouped._groups}")
print(f"Group IDs: {grouped._group_ids}")
print(f"n_groups: {grouped.n_groups}")

In [ ]:
# Test aggregations
result = grouped.agg(
    sum_qty=agg_sum('qty'),
    avg_price=agg_mean('price'),
    count=agg_count('qty'),
    device='cpu'
)
print("Result:")
print(result.to_pandas())

In [ ]:
# Test with GPU
result_gpu = grouped.agg(
    sum_qty=agg_sum('qty'),
    avg_price=agg_mean('price'),
    device='gpu'
)
print("GPU Result:")
print(result_gpu.to_pandas())

## Helper: group_by function

Standalone function for groupby (also added as method to MXFrame in core.py).

In [ ]:
#| export
def group_by(frame: MXFrame, *keys: str, device: DeviceType = "auto") -> MXGroupBy:
    """Group MXFrame by one or more columns.
    
    Args:
        frame: MXFrame to group
        *keys: Column names to group by
        device: Device for compute ('cpu', 'gpu', 'auto')
        
    Returns:
        MXGroupBy object with .agg() method
    
    Example:
        >>> result = group_by(frame, 'flag', 'status').agg(
        ...     sum_qty=agg_sum('qty')
        ... )
    """
    return MXGroupBy(frame, keys, device=device)

In [ ]:
# Test group_by function
result = group_by(frame, 'flag').agg(
    total_qty=agg_sum('qty'),
    n_rows=agg_count('qty'),
)
print("Group by single column:")
print(result.to_pandas())

## TPC-H Q1 Pattern Test

Test with the Q1 groupby pattern: group by (returnflag, linestatus) and compute aggregations.

In [ ]:
# TPC-H Q1 style test
import numpy as np

np.random.seed(42)
N = 10000

# Create test data similar to lineitem
flags = np.random.choice(['A', 'N', 'R'], N)
statuses = np.random.choice(['F', 'O'], N)
quantities = np.random.randint(1, 50, N).astype(np.float32)
prices = np.random.uniform(1.0, 100.0, N).astype(np.float32)
discounts = np.random.uniform(0.0, 0.1, N).astype(np.float32)

lineitem = MXFrame({
    'l_returnflag': flags.tolist(),
    'l_linestatus': statuses.tolist(),
    'l_quantity': quantities.tolist(),
    'l_extendedprice': prices.tolist(),
    'l_discount': discounts.tolist(),
})

print(f"lineitem: {lineitem.num_rows} rows")
print(f"Columns: {lineitem.column_names}")

In [ ]:
# Run Q1-style groupby with MAX Engine on CPU
import time

start = time.perf_counter()
q1_result = group_by(lineitem, 'l_returnflag', 'l_linestatus', device='cpu').agg(
    sum_qty=agg_sum('l_quantity'),
    sum_base_price=agg_sum('l_extendedprice'),
    count_order=agg_count('l_quantity'),
)
cpu_time = (time.perf_counter() - start) * 1000

print(f"Q1 GroupBy (CPU): {cpu_time:.2f}ms")
print(q1_result.to_pandas())

In [ ]:
# Run Q1-style groupby with MAX Engine on GPU
start = time.perf_counter()
q1_result_gpu = group_by(lineitem, 'l_returnflag', 'l_linestatus', device='gpu').agg(
    sum_qty=agg_sum('l_quantity'),
    sum_base_price=agg_sum('l_extendedprice'),
    count_order=agg_count('l_quantity'),
)
gpu_time = (time.perf_counter() - start) * 1000

print(f"Q1 GroupBy (GPU): {gpu_time:.2f}ms")
print(q1_result_gpu.to_pandas())

In [ ]:
#| hide
from nbdev import nbdev_export
nbdev_export()